# Step 4b – বাংলা TTS তুলনা: MMS-TTS বনাম Indic Parler-TTS (ন্যায্য মাপসহ)
**Runtime → Change runtime type → T4 GPU**, তারপর **Run all**।

কী হয়:
1. Step 4-এর একই ১০০টা MT বাংলা অনুবাদ (একই `random_state=13`), লেখা পরিষ্কার করে (`normalize_bn`)।
2. দুটো TTS দিয়ে কণ্ঠ বানানো: **MMS-TTS** (`facebook/mms-tts-ben`) আর **Indic Parler-TTS** (`ai4bharat/indic-parler-tts`, বক্তা Aditi, "clear standard Bengali")।
3. Whisper-large-v3-turbo দিয়ে আবার লিখিয়ে তিনভাবে CER/WER মাপা:
   - **raw** — যেমন লেখে তেমন
   - **mapped** — Whisper-এর হিন্দি (দেবনাগরী) অক্ষর বাংলায় বদলে
   - **constrained** — Whisper-কে লেখার সময় দেবনাগরী token ব্যবহার করতে না দিয়ে (ন্যায্য মাপ)
4. পাশাপাশি শোনার নমুনা, আর পরে MOS পরীক্ষার জন্য নাম লুকানো (blind) শোনার সেট।

ফল যায় `Sylheti_Thesis/sylheti_tts/run_compare/` — আগের কোনো ফল বদলায় না।

**আগে একবার (নিজে করবেন):** Indic Parler-TTS মডেলটা gated। huggingface.co/ai4bharat/indic-parler-tts খুলে শর্তে রাজি হোন, তারপর Colab-এর বাম পাশে 🔑 Secrets-এ নিজের Hugging Face token `HF_TOKEN` নামে রাখুন (notebook access চালু করে)। token কাউকে দেবেন না, কোডেও লিখবেন না।

## 1. Drive আর library

In [ ]:
from google.colab import drive
import os, sys, subprocess, importlib.util
os.environ["USE_TF"] = "0"; os.environ["USE_JAX"] = "0"   # keep transformers away from TensorFlow/JAX
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
N = 100              # same 100 test sentences as Step 4

# Indic Parler-TTS needs transformers 4.46.1 exactly. Its audio library pins protobuf < 5,
# but Colab's TensorFlow/TensorBoard files are built for protobuf 6.31+. So: install parler-tts,
# then put protobuf 6 back, then restart the runtime ONCE so nothing old stays in memory.
MARK = "/content/.parler_env_ready_v3"
if not os.path.exists(MARK) or importlib.util.find_spec("parler_tts") is None:
    pip = [sys.executable, "-m", "pip", "install", "-q"]
    subprocess.run(pip + ["git+https://github.com/huggingface/parler-tts.git", "transformers==4.46.1",
                          "jiwer==3.0.5", "soundfile", "librosa", "uroman"], check=True)
    subprocess.run(pip + ["protobuf>=6.31.1,<7"], check=True)   # pip may warn about descript-audiotools; that's OK
    open(MARK, "w").write("ok")
    print("✅ Packages installed. The runtime will now restart by itself.\n"
          "   When it reconnects, press Runtime → Run all again (this cell will then skip the install).")
    os.kill(os.getpid(), 9)

import google.protobuf, torch, transformers
assert transformers.__version__ == "4.46.1", f"transformers {transformers.__version__} — expected 4.46.1"
from transformers import VitsModel                      # fail early here if something is still mixed
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(0) if dev == "cuda" else "CPU (very slow for Parler)",
      "| transformers", transformers.__version__, "| protobuf", google.protobuf.__version__)

## 2. একই ১০০টা MT অনুবাদ

In [ ]:
import pandas as pd
D = "/content/drive/MyDrive/Sylheti_Thesis"
cands = [D + "/sylheti_mt_copy/nllb-sylheti", D + "/sylheti_mt/nllb-sylheti"]
MT_RUN = next((p for p in cands if os.path.exists(p + "/predictions_test.csv")), None)
assert MT_RUN, "predictions_test.csv পাওয়া যায়নি"
pred = pd.read_csv(MT_RUN + "/predictions_test.csv", encoding="utf-8-sig")
pred = pred.dropna(subset=["pred_bn", "pred_en"]).sample(min(N, len(pred)), random_state=13).reset_index(drop=True)
OUT = D + "/sylheti_tts/run_compare"
for sub in ["wav_mms", "wav_parler", "blind_listening"]:
    os.makedirs(f"{OUT}/{sub}", exist_ok=True)
print("MT results from:", MT_RUN, "|", len(pred), "sentences | example:", pred.pred_bn[0])

## 3. বাংলা লেখা পরিষ্কার করা (Step 4-এর মতোই)

In [ ]:
# ===== বাংলা লেখা পরিষ্কার করা (TTS-এ পাঠানোর আগে) =====
import re, unicodedata

_ONES = ("শূন্য এক দুই তিন চার পাঁচ ছয় সাত আট নয় দশ এগারো বারো তেরো চোদ্দ পনেরো ষোলো সতেরো আঠারো উনিশ "
         "বিশ একুশ বাইশ তেইশ চব্বিশ পঁচিশ ছাব্বিশ সাতাশ আটাশ উনত্রিশ ত্রিশ একত্রিশ বত্রিশ তেত্রিশ চৌত্রিশ পঁয়ত্রিশ ছত্রিশ সাঁইত্রিশ আটত্রিশ উনচল্লিশ "
         "চল্লিশ একচল্লিশ বিয়াল্লিশ তেতাল্লিশ চুয়াল্লিশ পঁয়তাল্লিশ ছেচল্লিশ সাতচল্লিশ আটচল্লিশ উনপঞ্চাশ পঞ্চাশ একান্ন বাহান্ন তিপ্পান্ন চুয়ান্ন পঞ্চান্ন ছাপ্পান্ন সাতান্ন আটান্ন উনষাট "
         "ষাট একষট্টি বাষট্টি তেষট্টি চৌষট্টি পঁয়ষট্টি ছেষট্টি সাতষট্টি আটষট্টি উনসত্তর সত্তর একাত্তর বাহাত্তর তিয়াত্তর চুয়াত্তর পঁচাত্তর ছিয়াত্তর সাতাত্তর আটাত্তর উনআশি "
         "আশি একাশি বিরাশি তিরাশি চুরাশি পঁচাশি ছিয়াশি সাতাশি আটাশি উননব্বই নব্বই একানব্বই বিরানব্বই তিরানব্বই চুরানব্বই পঁচানব্বই ছিয়ানব্বই সাতানব্বই আটানব্বই নিরানব্বই").split()
assert len(_ONES) == 100
_BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")

def num_to_bn(n: int) -> str:
    """পূর্ণ সংখ্যা → বাংলা কথায় (কোটি পর্যন্ত, তার বেশি হলে কোটি-র আগে আবার একই নিয়ম)"""
    if n < 100: return _ONES[n]
    parts = []
    for div, word in ((10**7, "কোটি"), (10**5, "লাখ"), (1000, "হাজার")):
        if n >= div:
            parts.append(num_to_bn(n // div) + " " + word); n %= div
    if n >= 100:
        parts.append(_ONES[n // 100] + "শো"); n %= 100
    if n: parts.append(_ONES[n])
    return " ".join(parts)

def _number(m):
    s = m.group(0).translate(_BN_DIGITS).replace(",", "")
    if "." in s:
        a, b = s.split(".", 1)
        return num_to_bn(int(a or 0)) + " দশমিক " + " ".join(_ONES[int(d)] for d in b)
    return num_to_bn(int(s))

def normalize_bn(text: str) -> str:
    t = unicodedata.normalize("NFC", str(text))
    t = t.replace("\u200c", "").replace("\u200d", "")                      # ZWNJ / ZWJ
    t = re.sub(r"[০-৯0-9]+(?:[,][০-৯0-9]{2,3})*(?:\.[০-৯0-9]+)?", _number, t)  # সংখ্যা → কথা
    t = t.replace("%", " শতাংশ").replace("&", " এবং ")
    t = re.sub(r"[/|\\]", " অথবা ", t)                                       # দিন/সপ্তাহ → দিন অথবা সপ্তাহ
    t = re.sub(r"[\"'“”‘’«»()\[\]{}<>*_#~^]", " ", t)                        # উদ্ধৃতি, বন্ধনী
    t = re.sub(r"\s*[-–—]\s*", " ", t)                                       # ড্যাশ → ফাঁকা
    t = re.sub(r"\.{2,}|…", "।", t)                                          # ... → দাঁড়ি
    t = re.sub(r"(?<=[\u0980-\u09FF])\.(?=\s|$)", "।", t)                    # বাংলা শব্দের পরে . → ।
    t = re.sub(r"\s*([।?!,;:])\s*", r"\1 ", t)                               # যতিচিহ্নের আগে ফাঁকা নয়
    t = re.sub(r"([।?!])[।?!\s]*", r"\1 ", t)                                # একাধিক দাঁড়ি → একটা
    t = re.sub(r"\s+", " ", t).strip()
    if t and t[-1] not in "।?!": t += "।"                                    # শেষে দাঁড়ি
    return t

def has_latin(text: str) -> bool:
    """ইংরেজি অক্ষর থাকলে True — বাংলা TTS এগুলো ঠিকমতো পড়তে পারে না"""
    return bool(re.search(r"[A-Za-z]", str(text)))
print(normalize_bn("আমি এখানে ৩ দিন/সপ্তাহের জন্য আছি"))

## 4. MMS-TTS দিয়ে কণ্ঠ

In [ ]:
import time, re, numpy as np, soundfile as sf, librosa
from transformers import VitsModel, AutoTokenizer
try:
    import uroman as ur; _uroman = ur.Uroman()
except Exception:
    _uroman = None
mms_tok = AutoTokenizer.from_pretrained("facebook/mms-tts-ben")
mms = VitsModel.from_pretrained("facebook/mms-tts-ben").to(dev).eval()

def mms_speak(text, seed=0):
    t = re.sub(r"\s+", " ", str(text)).strip()
    if getattr(mms_tok, "is_uroman", False) and _uroman is not None:
        t = _uroman.romanize_string(t)
    ids = mms_tok(t, return_tensors="pt").to(dev)
    torch.manual_seed(seed)
    with torch.no_grad():
        return mms(**ids).waveform[0].cpu().numpy(), mms.config.sampling_rate

rows = []
for i, r in pred.iterrows():
    text_tts = normalize_bn(r["pred_bn"])
    path = f"{OUT}/wav_mms/{i:03d}.wav"
    t0 = time.time(); wav, sr = mms_speak(text_tts); dt = time.time() - t0
    sf.write(path, wav, sr)
    rows.append({"id": i, "system": "MMS-TTS", "text": r["pred_bn"], "text_tts": text_tts,
                 "wav": path, "sec": len(wav) / sr, "gen_sec": dt})
mms_df = pd.DataFrame(rows)
print(f"MMS: {len(mms_df)} clips, {mms_df.sec.sum()/60:.1f} min, RTF {mms_df.gen_sec.sum()/mms_df.sec.sum():.3f}")
del mms; torch.cuda.empty_cache()

## 5. Indic Parler-TTS দিয়ে কণ্ঠ
বর্ণনা (`DESC`) বদলে কণ্ঠ বদলানো যায়। বাংলার জন্য মডেলের পাতায় সুপারিশ করা বক্তা: **Aditi** আর **Arjun**। T4-এ ১০০টা বাক্যে প্রায় ২০–৪০ মিনিট লাগতে পারে; আগের রানের ফাইল থাকলে সেগুলো আবার বানায় না।

In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami
login(token=userdata.get("HF_TOKEN"))          # reads your own secret; nothing is printed or saved in the notebook
print("Logged in to Hugging Face as:", whoami()["name"])

In [ ]:
from parler_tts import ParlerTTSForConditionalGeneration
PARLER = "ai4bharat/indic-parler-tts"
DESC = ("Aditi speaks clearly in standard Bengali at a moderate pace in a close-sounding environment, "
        "with very clear audio and no background noise.")
try:
    ptts = ParlerTTSForConditionalGeneration.from_pretrained(PARLER).to(dev).eval()
except Exception as e:
    raise SystemExit("Indic Parler-TTS লোড হয়নি। মডেলের পাতায় শর্তে রাজি হয়ে Colab Secrets-এ HF_TOKEN রেখেছেন কিনা দেখুন।\n" + str(e))
p_tok = AutoTokenizer.from_pretrained(PARLER)
d_tok = AutoTokenizer.from_pretrained(ptts.config.text_encoder._name_or_path)
desc_ids = d_tok(DESC, return_tensors="pt").to(dev)

def parler_speak(text, seed=0):
    prompt = p_tok(text, return_tensors="pt").to(dev)
    torch.manual_seed(seed)
    with torch.no_grad():
        gen = ptts.generate(input_ids=desc_ids.input_ids, attention_mask=desc_ids.attention_mask,
                            prompt_input_ids=prompt.input_ids, prompt_attention_mask=prompt.attention_mask)
    return gen.cpu().numpy().squeeze(), ptts.config.sampling_rate

rows = []
for _, r in mms_df.iterrows():
    path = f"{OUT}/wav_parler/{r.id:03d}.wav"
    if os.path.exists(path):                       # resume after a disconnect
        y, sr = sf.read(path); dt = np.nan
    else:
        t0 = time.time(); y, sr = parler_speak(r.text_tts); dt = time.time() - t0
        sf.write(path, y, sr)
    rows.append({"id": r.id, "system": "Indic Parler-TTS", "text": r.text, "text_tts": r.text_tts,
                 "wav": path, "sec": len(y) / sr, "gen_sec": dt})
    if r.id % 10 == 9: print("done", r.id + 1)
par_df = pd.DataFrame(rows)
g = par_df.dropna(subset=["gen_sec"])
print(f"Parler: {len(par_df)} clips, {par_df.sec.sum()/60:.1f} min" + (f", RTF {g.gen_sec.sum()/g.sec.sum():.3f}" if len(g) else ""))
del ptts; torch.cuda.empty_cache()

## 6. মূল্যায়ন: তিনভাবে মাপা
- **raw**: Whisper যেমন লেখে
- **mapped**: Whisper-এর দেবনাগরী অক্ষর বাংলায় বদলে
- **constrained**: Whisper-কে লেখার সময় দেবনাগরী token দেওয়া বন্ধ (সবচেয়ে ন্যায্য)

In [ ]:
import jiwer, unicodedata
from transformers import pipeline
asr = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3-turbo",
               torch_dtype=torch.float16 if dev == "cuda" else torch.float32, device=0 if dev == "cuda" else -1)
asr.model.generation_config.forced_decoder_ids = None
GK = {"language": "bengali", "task": "transcribe", "max_new_tokens": 440}

def norm(s):
    s = unicodedata.normalize("NFC", str(s)).lower()
    s = re.sub(r"[\"\'“”‘’(),.।?!:;\-/*…\[\]|]", " ", s)
    return re.sub(r"\s+", " ", s).strip()
def has_deva(s):
    return bool(re.search(r"[\u0900-\u097F]", str(s)))
def deva_to_beng(s):
    out = []
    for ch in str(s):
        c = ord(ch)
        if 0x0900 <= c <= 0x097F:
            b = chr(c + 0x80); out.append(b if unicodedata.name(b, "") else ch)
        else:
            out.append(ch)
    return "".join(out)

# token ids whose bytes contain a Devanagari character (UTF-8 starts E0 A4 / E0 A5)
def _bytes_to_unicode():          # GPT-2 / Whisper byte-level BPE alphabet
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs, n = bs[:], 0
    for b in range(256):
        if b not in bs:
            bs.append(b); cs.append(256 + n); n += 1
    return dict(zip(bs, map(chr, cs)))
byte_dec = {v: k for k, v in _bytes_to_unicode().items()}
tk = asr.tokenizer
deva_ids = []
for tok_str, idx in tk.get_vocab().items():
    try:
        bts = bytes(byte_dec[c] for c in tok_str)
    except KeyError:
        continue
    if b"\xe0\xa4" in bts or b"\xe0\xa5" in bts:
        deva_ids.append(idx)
base_suppress = list(asr.model.generation_config.suppress_tokens or [])
print("Devanagari tokens blocked in 'constrained' mode:", len(deva_ids))

def transcribe(paths, constrained=False):
    asr.model.generation_config.suppress_tokens = base_suppress + (deva_ids if constrained else [])
    audio = [librosa.load(p, sr=16000)[0] for p in paths]
    out = asr(audio, batch_size=8, generate_kwargs=GK)
    asr.model.generation_config.suppress_tokens = base_suppress
    return [o["text"] for o in out]

all_df = pd.concat([mms_df, par_df], ignore_index=True)
res = []
for sysname, s in all_df.groupby("system", sort=False):
    ref = [norm(t) for t in s.text_tts]
    raw = transcribe(s.wav.tolist())
    con = transcribe(s.wav.tolist(), constrained=True)
    hyp_raw = [norm(h) or "-" for h in raw]
    hyp_map = [norm(deva_to_beng(h)) or "-" for h in raw]
    hyp_con = [norm(h) or "-" for h in con]
    all_df.loc[s.index, "asr_raw"] = raw
    all_df.loc[s.index, "asr_constrained"] = con
    res.append({"system": sysname, "clips": len(s),
                "clips with Hindi letters (raw)": sum(has_deva(h) for h in raw),
                "clips with Hindi letters (constrained)": sum(has_deva(h) for h in con),
                "CER raw %": round(100 * jiwer.cer(ref, hyp_raw), 1), "WER raw %": round(100 * jiwer.wer(ref, hyp_raw), 1),
                "CER mapped %": round(100 * jiwer.cer(ref, hyp_map), 1), "WER mapped %": round(100 * jiwer.wer(ref, hyp_map), 1),
                "CER constrained %": round(100 * jiwer.cer(ref, hyp_con), 1), "WER constrained %": round(100 * jiwer.wer(ref, hyp_con), 1),
                "audio min": round(s.sec.sum() / 60, 1)})
res = pd.DataFrame(res)
res.to_csv(OUT + "/results_compare.csv", index=False)
all_df.to_csv(OUT + "/tts_compare_outputs.csv", index=False, encoding="utf-8-sig")
print(res.set_index("system").T.to_string())
print("\nsaved:", OUT + "/results_compare.csv")

## 7. পাশাপাশি শুনে দেখুন

In [ ]:
from IPython.display import Audio, display
import random
for i in random.sample(sorted(all_df.id.unique()), 4):           # new random sentences every run
    print(f"\n---- বাক্য #{i} ----")
    for sysname in ["MMS-TTS", "Indic Parler-TTS"]:
        r = all_df[(all_df.id == i) & (all_df.system == sysname)].iloc[0]
        print(f"{sysname:17s}| লেখা: {r.text_tts}\n{'':17s}| Whisper (constrained): {r.asr_constrained}")
        display(Audio(r.wav))

## 8. MOS পরীক্ষার জন্য blind শোনার সেট (ঐচ্ছিক)
২০টা বাক্য × ২টা মডেল = ৪০টা ফাইল, এলোমেলো নামে (`clip_01.wav` …)। কোনটা কোন মডেলের, তা শুধু `blind_key.csv`-এ থাকে — শ্রোতাদের সেটা দেবেন না। শ্রোতারা `blind_sheet.csv`-এ প্রতিটা ক্লিপকে ১–৫ নম্বর দেবেন (১ = খুব খারাপ, ৫ = একদম স্বাভাবিক)।

In [ ]:
import shutil
rng = random.Random(13)
ids20 = rng.sample(sorted(all_df.id.unique()), 20)
items = [all_df[(all_df.id == i) & (all_df.system == s)].iloc[0] for i in ids20 for s in ["MMS-TTS", "Indic Parler-TTS"]]
rng.shuffle(items)
key = []
for n, r in enumerate(items, 1):
    name = f"clip_{n:02d}.wav"
    shutil.copy(r.wav, f"{OUT}/blind_listening/{name}")
    key.append({"clip": name, "system": r.system, "sentence_id": r.id, "text": r.text_tts})
pd.DataFrame(key).to_csv(OUT + "/blind_key.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"clip": [k["clip"] for k in key], "text": [k["text"] for k in key], "score_1_to_5": ""}).to_csv(
    OUT + "/blind_listening/blind_sheet.csv", index=False, encoding="utf-8-sig")
print("blind set:", OUT + "/blind_listening  (40 clips + blind_sheet.csv)")

In [ ]:
#@title নিজে লিখুন → কণ্ঠ শুনুন
text = "tokenized input IDs sent to the MMS model's embedding layer are being interpreted as floating-point numbers instead of integer indexes. This is resolved by explicitly converting input_ids to torch.long (64-bit integers) inside the speak_mms function.  The previous edit introduced a SyntaxError by accidentally removing the backslash (\\) line continuation at the end of line 48 in cell 1tOZnsn4DIwr.  I will fix this by restoring the line continuation backslash so that Python correctly parses the multi-line conditional expression."  #@param {type:"string"}
language = "English"  #@param ["Bangla", "English"]
model_choice = "Both"  #@param ["MMS-TTS", "Indic Parler-TTS", "Both"]
save_to_drive = True  #@param {type:"boolean"}

import os, re, time, torch, numpy as np, soundfile as sf
from IPython.display import Audio, display
from transformers import VitsModel, AutoTokenizer
dev = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_DIR = "/content/drive/MyDrive/Sylheti_Thesis/sylheti_tts/my_text"

# load each model only once per session (kept in memory for the next run of this cell)
if "_tts_cache" not in globals():
    _tts_cache = {}

def _mms(lang):
    key = "mms_" + lang
    if key not in _tts_cache:
        name = "facebook/mms-tts-ben" if lang == "bn" else "facebook/mms-tts-eng"
        _tts_cache[key] = (AutoTokenizer.from_pretrained(name), VitsModel.from_pretrained(name).to(dev).eval())
    return _tts_cache[key]

def _parler():
    if "parler" not in _tts_cache:
        from parler_tts import ParlerTTSForConditionalGeneration
        P = "ai4bharat/indic-parler-tts"
        m = ParlerTTSForConditionalGeneration.from_pretrained(P).to(dev).eval()
        _tts_cache["parler"] = (m, AutoTokenizer.from_pretrained(P),
                                AutoTokenizer.from_pretrained(m.config.text_encoder._name_or_path))
    return _tts_cache["parler"]

def speak_mms(t, lang):
    tok, m = _mms(lang)
    if lang == "en":
        t = t.lower()
    ids = tok(t, return_tensors="pt")

    # Check if we generated any valid text tokens to avoid PyTorch empty tensor crashes
    if ids["input_ids"].shape[1] == 0:
         raise ValueError(f"Error: The input text contains no valid characters for the chosen {language} MMS model. Please write your text in the correct script or switch the language setting.")

    # Explicitly force input_ids to be LongTensor and move to device
    inputs = {"input_ids": ids["input_ids"].to(device=dev, dtype=torch.long)}
    if "attention_mask" in ids:
        inputs["attention_mask"] = ids["attention_mask"].to(dev)
    torch.manual_seed(0)
    with torch.no_grad():
        return m(**inputs).waveform[0].cpu().numpy(), m.config.sampling_rate

def speak_parler(t, lang):
    m, p_tok, d_tok = _parler()
    desc = ("Aditi speaks clearly in standard Bengali at a moderate pace in a close-sounding environment, "
            "with very clear audio and no background noise.") if lang == "bn" else \
           ("A female speaker speaks clearly in English at a moderate pace in a close-sounding environment, "
            "with very clear audio and no background noise.")
    d = d_tok(desc, return_tensors="pt").to(dev); p = p_tok(t, return_tensors="pt").to(dev)
    torch.manual_seed(0)
    with torch.no_grad():
        gen = m.generate(input_ids=d.input_ids, attention_mask=d.attention_mask,
                         prompt_input_ids=p.input_ids, prompt_attention_mask=p.attention_mask)
    return gen.cpu().numpy().squeeze(), m.config.sampling_rate

lang = "bn" if language == "Bangla" else "en"
t = re.sub(r"\s+", " ", text).strip()
assert t, "উপরের ফর্মে কিছু লিখুন"
if lang == "bn" and "normalize_bn" in globals():      # numbers → words, clean punctuation (from section 3)
    t = normalize_bn(t)
print("TTS-এ যাচ্ছে:", t)

systems = ["MMS-TTS", "Indic Parler-TTS"] if model_choice == "Both" else [model_choice]
for s in systems:
    t0 = time.time()
    try:
        wav, sr = speak_mms(t, lang) if s == "MMS-TTS" else speak_parler(t, lang)
        print(f"\n{s}  ({len(wav)/sr:.1f} s audio, made in {time.time()-t0:.1f} s)")
        display(Audio(wav, rate=sr))
        if save_to_drive:
            os.makedirs(SAVE_DIR, exist_ok=True)
            path = f"{SAVE_DIR}/{time.strftime('%Y%m%d_%H%M%S')}_{lang}_{s.split()[0].replace('-', '')}.wav"
            sf.write(path, wav, sr); print("saved:", path)
    except Exception as e:
        print(f"\n❌ {s} generation failed: {e}")

In [ ]:
import time, torch, soundfile as sf
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer
dev = "cuda"
name = "ai4bharat/indic-parler-tts"
m = ParlerTTSForConditionalGeneration.from_pretrained(name).to(dev).eval()
tok = AutoTokenizer.from_pretrained(name)
dtok = AutoTokenizer.from_pretrained(m.config.text_encoder._name_or_path)
desc = "Aditi speaks clearly in standard Bengali at a moderate pace with very clear audio."
texts = ["আমি আজকে বাজারে যাব।", "তুমি কেমন আছ?", "আজ আবহাওয়া খুব ভালো।", "আমার বাড়ি সিলেটে।",
         "সে স্কুলে পড়ে।", "আমরা কাল ঢাকা যাব।", "এটা আমার বই।", "তুমি কি খেয়েছ?",
         "বৃষ্টি হচ্ছে, ছাতা নিয়ে যাও।", "আমি তোমাকে পরে ফোন করব।"]
d = dtok(desc, return_tensors="pt").to(dev)
gen_t, audio_s = 0.0, 0.0
for t in texts:
    p = tok(t, return_tensors="pt").to(dev)
    torch.cuda.synchronize(); s = time.time()
    with torch.no_grad():
        w = m.generate(input_ids=d.input_ids, attention_mask=d.attention_mask,
                       prompt_input_ids=p.input_ids, prompt_attention_mask=p.attention_mask)
    torch.cuda.synchronize(); gen_t += time.time() - s
    audio_s += w.shape[-1] / m.config.sampling_rate
print(f"Indic Parler-TTS RTF = {gen_t/audio_s:.3f}  (GPU: {torch.cuda.get_device_name(0)})")